# 🎛️ Notebook 5 — Gradio Forensic Workstation
## Forensic Audio Authentication using Deep Learning

---

### 📋 Ce que fait ce notebook
1. Installe et configure Gradio
2. Charge le modèle ONNX (inférence rapide sans PyTorch)
3. Reconstruit le pipeline complet : audio → LFCC → modèle → verdict
4. Génère la heatmap Grad-CAM pour chaque analyse
5. Lance l'interface **Gradio forensique** avec lien public
6. Interface : upload audio → verdict + probabilité + heatmap + rapport

### ⏱️ Durée estimée : 10–15 minutes

### 📂 Inputs requis
```
best_model.pth    OU    forensic_audio.onnx
```

### 📂 Output
```
Lien public Gradio valable 72h  ← partager pour la démo
gradio_app.py                   ← script standalone
```

---
## CELLULE 1 — Installation des dépendances

In [3]:
import subprocess, sys

packages = [
    ('gradio',        'gradio'),
    ('onnxruntime',   'onnxruntime'),
    ('captum',        'captum'),
]

for import_name, pkg_name in packages:
    try:
        __import__(import_name)
        print(f'✅ {pkg_name} déjà installé')
    except ImportError:
        print(f'📦 Installation de {pkg_name}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg_name, '-q'],
                       capture_output=True)
        print(f'✅ {pkg_name} installé')

print('\n✅ Toutes les dépendances prêtes')

✅ gradio déjà installé
📦 Installation de onnxruntime...
✅ onnxruntime installé
📦 Installation de captum...
✅ captum installé

✅ Toutes les dépendances prêtes


---
## CELLULE 2 — Imports et configuration

In [1]:
import subprocess
subprocess.run(["pip", "install", "-q", "numpy==1.26.4", "pandas==2.2.2"], check=True)

CompletedProcess(args=['pip', 'install', '-q', 'numpy==1.26.4', 'pandas==2.2.2'], returncode=0)

In [2]:
import os, warnings, tempfile, time
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # backend non-interactif pour Gradio
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image
import io

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import gradio as gr

warnings.filterwarnings('ignore')

# ── Paramètres LFCC ───────────────────────────────────────────────────────────
SAMPLE_RATE   = 16000
CLIP_DURATION = 4
CLIP_SAMPLES  = SAMPLE_RATE * CLIP_DURATION
N_LFCC        = 20
N_FFT         = 400
HOP_LENGTH    = 160
N_FILTER      = 128
N_FRAMES      = 400

# ── Paramètres modèle ─────────────────────────────────────────────────────────
CNN_CHANNELS  = [32, 64, 128]
LSTM_HIDDEN   = 256
LSTM_LAYERS   = 2
LSTM_DROPOUT  = 0.3
CNN_DROPOUT   = 0.2
NUM_CLASSES   = 2

# ── Chemins ───────────────────────────────────────────────────────────────────
BASE    = Path('/kaggle/input/datasets/elmiz20042004')
OUTPUT  = Path('/kaggle/working')

MODEL_CANDIDATES = [
    Path('/kaggle/input/datasets/elmiz20042004/output1/checkpoints/best_model.pth'),
    Path('/kaggle/input/datasets/elmiz20042004/output1/models/forensic_audio.onnx'),
    BASE   / 'forensic-model/checkpoints/best_model.pth',
    BASE   / 'forensic-model/models/forensic_audio.onnx',
]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')
print(f'Gradio  : {gr.__version__}')
print('✅ Configuration OK')

Device  : cuda
Gradio  : 5.50.0
✅ Configuration OK


---
## CELLULE 3 — Définition du modèle CNN-BiLSTM et chargement

In [4]:
class CNNBiLSTM(nn.Module):
    def __init__(self,
                 n_lfcc=N_LFCC, n_frames=N_FRAMES,
                 cnn_channels=CNN_CHANNELS,
                 lstm_hidden=LSTM_HIDDEN, lstm_layers=LSTM_LAYERS,
                 lstm_dropout=LSTM_DROPOUT, cnn_dropout=CNN_DROPOUT,
                 num_classes=NUM_CLASSES):
        super().__init__()
        self.n_lfcc   = n_lfcc
        self.n_frames = n_frames

        self.cnn = nn.Sequential(
            nn.Conv2d(1, cnn_channels[0], 3, padding=1),
            nn.BatchNorm2d(cnn_channels[0]), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(cnn_channels[0], cnn_channels[1], 3, padding=1),
            nn.BatchNorm2d(cnn_channels[1]), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(cnn_channels[1], cnn_channels[2], 3, padding=1),
            nn.BatchNorm2d(cnn_channels[2]), nn.ReLU(True),
            nn.Dropout2d(cnn_dropout),
        )
        self._cnn_out = self._cnn_size(n_lfcc, n_frames)
        self.bilstm = nn.LSTM(
            self._cnn_out, lstm_hidden, lstm_layers,
            batch_first=True, bidirectional=True,
            dropout=lstm_dropout if lstm_layers > 1 else 0
        )
        self.classifier = nn.Sequential(
            nn.Linear(lstm_hidden*2, 128), nn.ReLU(True),
            nn.Dropout(0.3), nn.Linear(128, num_classes)
        )

    def _cnn_size(self, h, w):
        with torch.no_grad():
            x = torch.zeros(1,1,h,w)
            o = self.cnn(x)
            return o.shape[1] * o.shape[2]

    def forward(self, x):
        b = x.shape[0]
        o = self.cnn(x)
        _, c, h, w = o.shape
        s = o.permute(0,3,1,2).contiguous().view(b, w, c*h)
        _, (hn, _) = self.bilstm(s)
        f = torch.cat([hn[-2], hn[-1]], 1)
        return self.classifier(f)


# ── Chercher et charger le modèle ─────────────────────────────────────────────
model      = None
onnx_sess  = None
model_type = None

for candidate in MODEL_CANDIDATES:
    if not Path(candidate).exists():
        continue
    if str(candidate).endswith('.pth'):
        try:
            model = CNNBiLSTM().to(device)
            ckpt  = torch.load(candidate, map_location=device)
            model.load_state_dict(ckpt['model_state'])
            model.eval()
            model_type = 'pytorch'
            print(f'✅ Modèle PyTorch chargé : {candidate}')
            break
        except Exception as e:
            print(f'⚠️  Erreur chargement .pth : {e}')

    elif str(candidate).endswith('.onnx'):
        try:
            import onnxruntime as ort
            onnx_sess  = ort.InferenceSession(str(candidate))
            model_type = 'onnx'
            print(f'✅ Modèle ONNX chargé : {candidate}')
            break
        except Exception as e:
            print(f'⚠️  Erreur chargement .onnx : {e}')

if model_type is None:
    print('⚠️  Aucun modèle trouvé — utilisation de poids aléatoires (démo seulement)')
    model      = CNNBiLSTM().to(device)
    model.eval()
    model_type = 'pytorch_random'

print(f'   Mode : {model_type}')

✅ Modèle PyTorch chargé : /kaggle/input/datasets/elmiz20042004/output1/checkpoints/best_model.pth
   Mode : pytorch


---
## CELLULE 4 — Pipeline audio → LFCC → prédiction

In [5]:
# ── Transform LFCC ────────────────────────────────────────────────────────────
lfcc_transform = T.LFCC(
    sample_rate=SAMPLE_RATE, n_filter=N_FILTER, n_lfcc=N_LFCC,
    speckwargs={'n_fft': N_FFT, 'hop_length': HOP_LENGTH, 'win_length': N_FFT}
)


def audio_to_lfcc_tensor(audio_path):
    """
    Fichier audio → Tensor (1,1,20,400) prêt pour le modèle.
    Accepte .wav, .flac, .mp3, .ogg
    """
    wav, sr = torchaudio.load(str(audio_path))

    # Rééchantillonnage
    if sr != SAMPLE_RATE:
        wav = T.Resample(sr, SAMPLE_RATE)(wav)

    # Mono
    if wav.shape[0] > 1:
        wav = wav.mean(0, keepdim=True)

    # Longueur fixe 4s
    n = wav.shape[1]
    if n >= CLIP_SAMPLES:
        s = (n - CLIP_SAMPLES) // 2
        wav = wav[:, s:s+CLIP_SAMPLES]
    else:
        wav = F.pad(wav, (0, CLIP_SAMPLES - n))

    # Normalisation amplitude
    rms = wav.pow(2).mean().sqrt()
    if rms > 1e-8:
        wav = wav * (10**(-20/20) / rms)
    wav = wav.clamp(-1.0, 1.0)

    # LFCC
    with torch.no_grad():
        lfcc = lfcc_transform(wav).squeeze(0)  # (20, ~400)

    # CMVN
    lfcc = (lfcc - lfcc.mean(1, keepdim=True)) / (lfcc.std(1, keepdim=True) + 1e-8)

    # Ajuster frames
    nf = lfcc.shape[1]
    if nf >= N_FRAMES:
        s = (nf - N_FRAMES) // 2
        lfcc = lfcc[:, s:s+N_FRAMES]
    else:
        lfcc = F.pad(lfcc, (0, N_FRAMES - nf))

    return lfcc.unsqueeze(0).unsqueeze(0), wav  # (1,1,20,400), (1,64000)


def run_inference(lfcc_tensor):
    """
    Tensor (1,1,20,400) → (prob_authentic, prob_fake)
    Supporte PyTorch et ONNX.
    """
    if model_type == 'onnx':
        inp    = {'lfcc_input': lfcc_tensor.cpu().numpy()}
        logits = onnx_sess.run(None, inp)[0]       # (1, 2)
        logits = torch.tensor(logits)
    else:
        with torch.no_grad():
            logits = model(lfcc_tensor.to(device)).cpu()

    probs = F.softmax(logits, dim=1)[0]
    return float(probs[0]), float(probs[1])  # prob_auth, prob_fake


print('✅ Pipeline audio → LFCC → prédiction défini')

✅ Pipeline audio → LFCC → prédiction défini


---
## CELLULE 5 — Grad-CAM pour l'interface

In [8]:
class GradCAMApp:
    def __init__(self, mdl):
        self.model       = mdl
        self.gradients   = None
        self.activations = None
        conv_layers      = [m for m in mdl.cnn.modules() if isinstance(m, nn.Conv2d)]
        if conv_layers:
            self.target = conv_layers[-1]
            self.target.register_forward_hook(
                lambda m, i, o: setattr(self, 'activations', o.detach())
            )
            self.target.register_full_backward_hook(
                lambda m, gi, go: setattr(self, 'gradients', go[0].detach())
            )
            self.available = True
        else:
            self.available = False

    def compute(self, lfcc_tensor, target_class=1):
        if not self.available:
            return np.zeros((N_LFCC, N_FRAMES))
        
        # ✅ CORRECTIF : train() pour autoriser le backward du BiLSTM sur cuDNN
        self.model.train()
        
        inp    = lfcc_tensor.to(device).requires_grad_(True)
        logits = self.model(inp)
        self.model.zero_grad()
        logits[0, target_class].backward()
        
        weights = self.gradients.mean([2,3], keepdim=True)
        cam     = F.relu((weights * self.activations).sum(1, keepdim=True))
        cam     = F.interpolate(cam, (N_LFCC, N_FRAMES), mode='bilinear', align_corners=False)
        cam     = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        
        # ✅ Revenir en eval après le backward
        self.model.eval()
        return cam

---
## CELLULE 6 — Fonction de génération de la figure forensique

In [12]:
def generate_forensic_figure(audio_path, prob_auth, prob_fake, lfcc_np, cam, waveform_np):
    """
    Génère la figure forensique complète retournée par Gradio.
    Retourne une image PIL.
    """
    pred_label    = 'MANIPULÉ ⚠️'    if prob_fake >= 0.5 else 'AUTHENTIQUE ✅'
    verdict_color = '#C55A11'          if prob_fake >= 0.5 else '#375623'
    confidence    = max(prob_fake, prob_auth)

    # Localisation temporelle du pic d'anomalie
    peak_frame = int(np.argmax(cam.mean(axis=0)))
    peak_time  = peak_frame * HOP_LENGTH / SAMPLE_RATE

    fig = plt.figure(figsize=(14, 9))
    gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.5, wspace=0.35)

    fig.suptitle(
        f'🔬 Analyse Forensique Audio\n'
        f'VERDICT : {pred_label}  |  '
        f'Probabilité : {prob_fake:.1%}  |  '
        f'Confiance : {confidence:.1%}',
        fontsize=13, fontweight='bold', color=verdict_color, y=1.01
    )

    t_axis = np.linspace(0, CLIP_DURATION, len(waveform_np))

    # ── Panneau 1 : Waveform ─────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, :])
    ax1.plot(t_axis, waveform_np, color='#2E75B6', linewidth=0.4, alpha=0.85)
    ax1.axvline(peak_time, color='red', linewidth=2.0,
                label=f'Anomalie détectée : {peak_time:.2f}s')
    # Zone d'anomalie ±0.2s
    ax1.axvspan(max(0, peak_time-0.2), min(CLIP_DURATION, peak_time+0.2),
                alpha=0.12, color='red', label='Zone suspecte (±200ms)')
    ax1.set_xlabel('Temps (secondes)')
    ax1.set_ylabel('Amplitude')
    ax1.set_title('Signal audio (waveform)', fontweight='bold')
    ax1.legend(fontsize=9, loc='upper right')
    ax1.grid(True, alpha=0.2)

    # ── Panneau 2 : Matrice LFCC ─────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[1, 0])
    im2 = ax2.imshow(lfcc_np, aspect='auto', origin='lower',
                      cmap='inferno', vmin=-3, vmax=3)
    ax2.set_title('Matrice LFCC (entrée CNN)', fontweight='bold')
    ax2.set_xlabel('Frames (×10ms)')
    ax2.set_ylabel('Coefficient LFCC')
    plt.colorbar(im2, ax=ax2, shrink=0.85)

    # ── Panneau 3 : Grad-CAM ─────────────────────────────────────────────────
    ax3 = fig.add_subplot(gs[1, 1])
    ax3.imshow(lfcc_np, aspect='auto', origin='lower',
               cmap='gray', alpha=0.55, vmin=-3, vmax=3)
    im3 = ax3.imshow(cam, aspect='auto', origin='lower',
                      cmap='Reds', alpha=0.65, vmin=0, vmax=1)
    ax3.axvline(peak_frame, color='cyan', linewidth=2.0,
                label=f'Pic : {peak_time:.2f}s')
    ax3.set_title('Heatmap Grad-CAM\n(rouge = zone déclenchant la détection)',
                   fontweight='bold')
    ax3.set_xlabel('Frames (×10ms)')
    ax3.set_ylabel('Coefficient LFCC')
    ax3.legend(fontsize=8, loc='upper right')
    plt.colorbar(im3, ax=ax3, shrink=0.85)

    # ── Panneau 4 : Profil spectral ───────────────────────────────────────────
    ax4 = fig.add_subplot(gs[2, 0])
    cam_coeff   = cam.mean(axis=1)
    bar_colors  = ['#C55A11' if v > cam_coeff.mean() else '#2E75B6'
                   for v in cam_coeff]
    ax4.barh(range(N_LFCC), cam_coeff, color=bar_colors,
              edgecolor='white', alpha=0.85)
    ax4.axhline(y=14, color='red', linestyle='--', alpha=0.5,
                label='Hautes fréq. (deepfake zone)')
    ax4.set_xlabel('Activation Grad-CAM moyenne')
    ax4.set_ylabel('Coefficient LFCC')
    ax4.set_title('Profil spectral des anomalies', fontweight='bold')
    ax4.legend(fontsize=8)

    # ── Panneau 5 : Rapport textuel ───────────────────────────────────────────
    ax5 = fig.add_subplot(gs[2, 1])
    ax5.axis('off')

    high_act = cam[15:, :].mean()
    low_act  = cam[:5,  :].mean()

    if prob_fake >= 0.5:
        if high_act > low_act:
            interp = 'Artefacts hautes fréquences\n→ Signature vocoder deepfake\n  (HiFi-GAN / MelGAN / WaveGlow)'
        else:
            interp = 'Perturbation basses fréquences\n→ Signature de splice/coupure\n  (discontinuité du fondamental)'
    else:
        interp = 'Distribution normale\n→ Pas d\'anomalie détectée\n  Enregistrement cohérent'

    report = (
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'  RAPPORT FORENSIQUE\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
        f'\n'
        f'  Verdict      : {pred_label}\n'
        f'  Probabilité  : {prob_fake:.1%}\n'
        f'  Confiance    : {confidence:.1%}\n'
        f'\n'
        f'  Anomalie à   : {peak_time:.3f}s\n'
        f'  Frame LFCC   : {peak_frame}\n'
        f'\n'
        f'  Analyse      :\n'
        f'  {interp}\n'
        f'\n'
        f'  Modèle       : CNN-BiLSTM\n'
        f'  Features     : LFCC (20 coeff.)\n'
        f'  XAI          : Grad-CAM\n'
        f'━━━━━━━━━━━━━━━━━━━━━━━━━━━'
    )

    ax5.text(0.05, 0.95, report,
              transform=ax5.transAxes,
              fontsize=9, verticalalignment='top',
              fontfamily='monospace',
              bbox=dict(boxstyle='round', facecolor='#F8F8F8',
                        edgecolor=verdict_color, linewidth=2, alpha=0.95))
    ax5.set_title('Rapport automatique', fontweight='bold')

    plt.tight_layout()

    # Convertir en image PIL pour Gradio
    buf = io.BytesIO()
    plt.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    buf.seek(0)
    img = Image.open(buf).copy()
    plt.close()
    buf.close()
    return img


print('✅ generate_forensic_figure() définie')

✅ generate_forensic_figure() définie


---
## CELLULE 7 — Fonction principale d'analyse (appelée par Gradio)

In [13]:
def analyze_audio(audio_input):
    """
    Fonction principale appelée par Gradio.

    Args:
        audio_input : chemin vers le fichier audio uploadé (str)

    Returns:
        verdict_text : str — texte du verdict
        prob_label   : dict — pour gr.Label
        figure_img   : PIL.Image — figure forensique complète
        report_text  : str — rapport textuel
    """
    if audio_input is None:
        return (
            '⚠️ Aucun fichier uploadé',
            {'Authentique': 0.5, 'Manipulé': 0.5},
            None,
            'Veuillez uploader un fichier audio.'
        )

    try:
        t_start = time.time()

        # ── Étape 1 : Audio → LFCC ────────────────────────────────────────────
        lfcc_tensor, waveform = audio_to_lfcc_tensor(audio_input)
        lfcc_np               = lfcc_tensor.squeeze().cpu().numpy()  # (20, 400)
        waveform_np           = waveform.squeeze().cpu().numpy()     # (64000,)

        # ── Étape 2 : Prédiction ─────────────────────────────────────────────
        prob_auth, prob_fake = run_inference(lfcc_tensor)
        pred_class           = int(prob_fake >= 0.5)

        # ── Étape 3 : Grad-CAM ───────────────────────────────────────────────
        if gradcam_app is not None and gradcam_app.available:
            cam = gradcam_app.compute(lfcc_tensor, target_class=1)
        else:
            # Fallback : gradient simple sur l'input
            # inp = lfcc_tensor.to(device).requires_grad_(True)
            if model is not None:
                model.train()  # ✅ CORRECTIF ici aussi
                inp = lfcc_tensor.to(device).requires_grad_(True)
                logits = model(inp)
                model.zero_grad()
                logits[0, 1].backward()
                cam = inp.grad.squeeze().abs().cpu().numpy()
                cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                model.eval()  # ✅ Revenir en eval
            else:
                cam = np.random.rand(N_LFCC, N_FRAMES) * 0.1

        # ── Étape 4 : Figure forensique ──────────────────────────────────────
        figure_img = generate_forensic_figure(
            audio_path  = audio_input,
            prob_auth   = prob_auth,
            prob_fake   = prob_fake,
            lfcc_np     = lfcc_np,
            cam         = cam,
            waveform_np = waveform_np
        )

        # ── Étape 5 : Outputs Gradio ─────────────────────────────────────────
        elapsed = time.time() - t_start

        peak_frame = int(np.argmax(cam.mean(axis=0)))
        peak_time  = peak_frame * HOP_LENGTH / SAMPLE_RATE
        high_act   = cam[15:, :].mean()
        low_act    = cam[:5,  :].mean()

        if pred_class == 1:
            verdict_emoji = '🚨'
            verdict_text  = f'{verdict_emoji} MANIPULÉ ({prob_fake:.1%})'
            manip_type    = 'deepfake vocoder' if high_act > low_act else 'splice/coupure'
        else:
            verdict_emoji = '✅'
            verdict_text  = f'{verdict_emoji} AUTHENTIQUE ({prob_auth:.1%})'
            manip_type    = 'N/A'

        prob_label = {
            'Authentique ✅': round(prob_auth, 4),
            'Manipulé ⚠️'   : round(prob_fake, 4),
        }

        report_text = (
            f'VERDICT : {verdict_text}\n'
            f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
            f'Probabilité manipulation : {prob_fake:.2%}\n'
            f'Probabilité authentique  : {prob_auth:.2%}\n'
            f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
            f'Anomalie localisée à     : {peak_time:.3f}s\n'
            f'Frame LFCC               : {peak_frame}/{N_FRAMES}\n'
            f'Type de manipulation     : {manip_type}\n'
            f'━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━\n'
            f'Temps d\'analyse          : {elapsed:.2f}s\n'
            f'Modèle                   : CNN-BiLSTM ({model_type})\n'
            f'Features                 : LFCC 20 coeff. CMVN\n'
            f'XAI                      : Grad-CAM\n'
        )

        return verdict_text, prob_label, figure_img, report_text

    except Exception as e:
        error_msg = f'❌ Erreur lors de l\'analyse : {str(e)}'
        return (
            error_msg,
            {'Authentique': 0.5, 'Manipulé': 0.5},
            None,
            error_msg
        )


print('✅ analyze_audio() définie')
print('   Inputs  : fichier audio')
print('   Outputs : verdict_text, prob_label, figure_img, report_text')

✅ analyze_audio() définie
   Inputs  : fichier audio
   Outputs : verdict_text, prob_label, figure_img, report_text


---
## CELLULE 8 — Test de la fonction avant de lancer Gradio

In [14]:
# ── Réinitialiser GradCAM avec le correctif ──────────────────────────────────
class GradCAMApp:
    def __init__(self, mdl):
        self.model       = mdl
        self.gradients   = None
        self.activations = None
        conv_layers      = [m for m in mdl.cnn.modules() if isinstance(m, nn.Conv2d)]
        if conv_layers:
            self.target = conv_layers[-1]
            self.target.register_forward_hook(
                lambda m, i, o: setattr(self, 'activations', o.detach())
            )
            self.target.register_full_backward_hook(
                lambda m, gi, go: setattr(self, 'gradients', go[0].detach())
            )
            self.available = True
        else:
            self.available = False

    def compute(self, lfcc_tensor, target_class=1):
        if not self.available:
            return np.zeros((N_LFCC, N_FRAMES))
        self.model.train()   # ✅ CORRECTIF cuDNN
        inp    = lfcc_tensor.to(device).requires_grad_(True)
        logits = self.model(inp)
        self.model.zero_grad()
        logits[0, target_class].backward()
        weights = self.gradients.mean([2,3], keepdim=True)
        cam     = F.relu((weights * self.activations).sum(1, keepdim=True))
        cam     = F.interpolate(cam, (N_LFCC, N_FRAMES), mode='bilinear', align_corners=False)
        cam     = cam.squeeze().cpu().numpy()
        if cam.max() > 0:
            cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        self.model.eval()    # ✅ Revenir en eval
        return cam

# Réinstancier avec le modèle déjà chargé
gradcam_app = GradCAMApp(model)
print(f'✅ GradCAM réinitialisé : {gradcam_app.available}')

✅ GradCAM réinitialisé : True


In [15]:
# ── Trouver un fichier test ───────────────────────────────────────────────────
test_file = None

# Chercher dans WaveFake
if BASE.exists():
    wavefake = BASE / 'forensic-audio-dataset/generated_audio/generated_audio'
    if wavefake.exists():
        for vdir in sorted(wavefake.iterdir()):
            if vdir.is_dir():
                wavs = sorted(vdir.glob('*.wav'))
                if wavs:
                    test_file = str(wavs[0])
                    break

# Chercher dans spliced
if test_file is None:
    spliced = BASE / 'audio-preprocessing/spliced'
    if spliced.exists():
        wavs = sorted(spliced.glob('*.wav'))
        if wavs:
            test_file = str(wavs[0])

if test_file:
    print(f'🧪 Test sur : {Path(test_file).name}')
    verdict, probs, fig_img, report = analyze_audio(test_file)

    print(f'\n   Verdict : {verdict}')
    print(f'   Probs   : {probs}')
    print(f'   Figure  : {"OK" if fig_img is not None else "ERREUR"}')
    print(f'   Rapport :\n{report}')

    if fig_img:
        plt.figure(figsize=(14, 9))
        plt.imshow(fig_img)
        plt.axis('off')
        plt.title('Test figure forensique', fontsize=10)
        plt.tight_layout()
        plt.show()
        print('✅ Test réussi — Gradio peut être lancé')
    else:
        print('⚠️  Figure non générée — vérifier les erreurs')
else:
    print('⚠️  Aucun fichier test trouvé — Gradio sera lancé quand même')

🧪 Test sur : BASIC5000_0001_gen.wav

   Verdict : 🚨 MANIPULÉ (100.0%)
   Probs   : {'Authentique ✅': 0.0005, 'Manipulé ⚠️': 0.9995}
   Figure  : OK
   Rapport :
VERDICT : 🚨 MANIPULÉ (100.0%)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Probabilité manipulation : 99.95%
Probabilité authentique  : 0.05%
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Anomalie localisée à     : 2.220s
Frame LFCC               : 222/400
Type de manipulation     : splice/coupure
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Temps d'analyse          : 6.00s
Modèle                   : CNN-BiLSTM (pytorch)
Features                 : LFCC 20 coeff. CMVN
XAI                      : Grad-CAM

✅ Test réussi — Gradio peut être lancé


---
## CELLULE 9 — Interface Gradio complète

> Cette cellule lance l'interface publique.  
> Un lien **https://xxx.gradio.live** sera affiché — valable 72h.  
> Partager ce lien pour la démo.

In [16]:
# ── Exemples audio pour la démo ───────────────────────────────────────────────
example_files = []
wavefake_dir  = BASE / 'forensic-audio-dataset/generated_audio/generated_audio'
spliced_dir   = BASE / 'audio-preprocessing/spliced'

if wavefake_dir.exists():
    for vdir in sorted(wavefake_dir.iterdir()):
        if not vdir.is_dir(): continue
        wavs = sorted(vdir.glob('*.wav'))
        for w in wavs[:2]:
            example_files.append([str(w)])
        if len(example_files) >= 4:
            break

if spliced_dir.exists():
    for w in sorted(spliced_dir.glob('splice_*.wav'))[:2]:
        example_files.append([str(w)])

print(f'Exemples disponibles : {len(example_files)}')

# ── Construction de l'interface Gradio ────────────────────────────────────────
with gr.Blocks(
    title='Forensic Audio Authentication',
    theme=gr.themes.Soft(
        primary_hue='blue',
        secondary_hue='orange',
    ),
    css='''
        .verdict-box { font-size: 1.4em; font-weight: bold; text-align: center; }
        .title-box   { text-align: center; margin-bottom: 10px; }
    '''
) as demo:

    # ── En-tête ───────────────────────────────────────────────────────────────
    gr.HTML("""
    <div class='title-box'>
        <h1>🎙️ Forensic Audio Authentication</h1>
        <p style='color: #666; font-size: 1.1em;'>
            Détection de manipulation audio par Deep Learning (CNN-BiLSTM + LFCC + Grad-CAM)
        </p>
        <p style='color: #888; font-size: 0.9em;'>
            Uploadez un fichier audio (.wav, .flac, .mp3) pour analyser son authenticité
        </p>
    </div>
    """)

    with gr.Row():
        # ── Colonne gauche : Input ────────────────────────────────────────────
        with gr.Column(scale=1):
            gr.Markdown('### 📂 Fichier audio à analyser')

            audio_input = gr.Audio(
                label   = 'Upload audio',
                type    = 'filepath',
                sources = ['upload', 'microphone'],
            )

            analyze_btn = gr.Button(
                '🔍 Analyser',
                variant = 'primary',
                size    = 'lg'
            )

            gr.Markdown('### 📊 Verdict')
            verdict_output = gr.Textbox(
                label       = 'Résultat',
                interactive = False,
                lines       = 1,
                elem_classes= ['verdict-box']
            )

            prob_output = gr.Label(
                label     = 'Probabilités',
                num_top_classes = 2
            )

            gr.Markdown('### 📋 Rapport détaillé')
            report_output = gr.Textbox(
                label       = 'Rapport forensique',
                interactive = False,
                lines       = 15,
                max_lines   = 20
            )

        # ── Colonne droite : Visualisation ────────────────────────────────────
        with gr.Column(scale=2):
            gr.Markdown('### 🔬 Analyse visuelle (Waveform + LFCC + Grad-CAM)')
            figure_output = gr.Image(
                label  = 'Analyse forensique complète',
                type   = 'pil',
                height = 600
            )

    # ── Exemples ──────────────────────────────────────────────────────────────
    if example_files:
        gr.Markdown('### 📁 Exemples de clips à tester')
        gr.Examples(
            examples    = example_files,
            inputs      = [audio_input],
            label       = 'Cliquer sur un exemple pour l\'analyser'
        )

    # ── Informations ──────────────────────────────────────────────────────────
    with gr.Accordion('ℹ️ Comment interpréter les résultats', open=False):
        gr.Markdown("""
        ### Guide d'interprétation

        **Verdict AUTHENTIQUE ✅**
        - Le modèle n'a détecté aucune anomalie significative dans le signal
        - La distribution LFCC est cohérente avec un enregistrement naturel
        - La heatmap Grad-CAM montre une activation diffuse, sans pic suspect

        **Verdict MANIPULÉ ⚠️**
        - Une ou plusieurs anomalies ont été détectées dans le signal
        - La ligne rouge sur le waveform indique le moment suspect
        - Les zones rouges sur la heatmap Grad-CAM montrent les fréquences anormales

        **Interprétation de la heatmap :**
        - **Hautes fréquences actives (coefficients 15-19)** → signature de vocoder deepfake
        - **Basses fréquences actives (coefficients 0-4)** → discontinuité de splice
        - **Activation concentrée verticalement** → artefact ponctuel dans le temps

        **Métriques du modèle :**
        - Architecture : CNN-BiLSTM (20 epochs)
        - Features : LFCC 20 coefficients + CMVN
        - Datasets : ASVspoof2019 LA + WaveFake + splices simulés
        - XAI : Grad-CAM sur dernière couche Conv2D
        """)

    # ── Connexion bouton → fonction ───────────────────────────────────────────
    analyze_btn.click(
        fn      = analyze_audio,
        inputs  = [audio_input],
        outputs = [verdict_output, prob_output, figure_output, report_output]
    )

    # Lancer automatiquement quand un exemple est sélectionné
    audio_input.change(
        fn      = analyze_audio,
        inputs  = [audio_input],
        outputs = [verdict_output, prob_output, figure_output, report_output]
    )

print('✅ Interface Gradio construite')
print('🚀 Lancement...')
print()

# ── Lancement avec lien public ────────────────────────────────────────────────
demo.launch(
    share          = True,     # génère un lien public https://xxx.gradio.live
    debug          = False,
    show_error     = True,
    quiet          = False,
    inbrowser      = False,    # ne pas ouvrir de navigateur (Kaggle)
)

Exemples disponibles : 6
✅ Interface Gradio construite
🚀 Lancement...

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://40ef924c3f3d26b7a2.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## CELLULE 10 — Export du script Gradio standalone (gradio_app.py)

In [17]:
# ── Exporter un script Python autonome ───────────────────────────────────────
# Ce script peut tourner en dehors de Kaggle
# Il suffit d'avoir best_model.pth dans le même dossier

app_script = '''
"""
Forensic Audio Authentication — Gradio App standalone
Usage : python gradio_app.py
Requires : best_model.pth dans le meme dossier
"""

import warnings, io, time
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
import gradio as gr

warnings.filterwarnings("ignore")

SAMPLE_RATE = 16000
CLIP_SAMPLES= 64000
N_LFCC      = 20
N_FFT       = 400
HOP_LENGTH  = 160
N_FILTER    = 128
N_FRAMES    = 400
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class CNNBiLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(True), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(True), nn.Dropout2d(0.2),
        )
        with torch.no_grad():
            o = self.cnn(torch.zeros(1,1,N_LFCC,N_FRAMES))
            cnn_out = o.shape[1] * o.shape[2]
        self.bilstm = nn.LSTM(cnn_out,256,2,batch_first=True,bidirectional=True,dropout=0.3)
        self.classifier = nn.Sequential(
            nn.Linear(512,128), nn.ReLU(True), nn.Dropout(0.3), nn.Linear(128,2)
        )
    def forward(self, x):
        b=x.shape[0]; o=self.cnn(x); _,c,h,w=o.shape
        s=o.permute(0,3,1,2).contiguous().view(b,w,c*h)
        _,(hn,_)=self.bilstm(s)
        return self.classifier(torch.cat([hn[-2],hn[-1]],1))


model = CNNBiLSTM().to(device)
ckpt_path = Path("best_model.pth")
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    print(f"Modele charge depuis {ckpt_path}")
else:
    print("ATTENTION : best_model.pth non trouve — poids aleatoires")
model.eval()

lfcc_tf = T.LFCC(SAMPLE_RATE,N_FILTER,N_LFCC,
                  speckwargs={"n_fft":N_FFT,"hop_length":HOP_LENGTH,"win_length":N_FFT})

def process(path):
    wav,sr = torchaudio.load(str(path))
    if sr!=SAMPLE_RATE: wav=T.Resample(sr,SAMPLE_RATE)(wav)
    if wav.shape[0]>1: wav=wav.mean(0,keepdim=True)
    n=wav.shape[1]
    if n>=CLIP_SAMPLES: s=(n-CLIP_SAMPLES)//2; wav=wav[:,s:s+CLIP_SAMPLES]
    else: wav=F.pad(wav,(0,CLIP_SAMPLES-n))
    rms=wav.pow(2).mean().sqrt()
    if rms>1e-8: wav=wav*(10**(-20/20)/rms)
    with torch.no_grad(): lfcc=lfcc_tf(wav).squeeze(0)
    lfcc=(lfcc-lfcc.mean(1,keepdim=True))/(lfcc.std(1,keepdim=True)+1e-8)
    nf=lfcc.shape[1]
    if nf>=N_FRAMES: s=(nf-N_FRAMES)//2; lfcc=lfcc[:,s:s+N_FRAMES]
    else: lfcc=F.pad(lfcc,(0,N_FRAMES-nf))
    return lfcc.unsqueeze(0).unsqueeze(0), wav

def analyze(audio):
    if audio is None: return "Aucun fichier",{"Authentique":0.5,"Manipule":0.5},None,""
    lfcc_t,wav=process(audio)
    with torch.no_grad():
        probs=F.softmax(model(lfcc_t.to(device)),1)[0].cpu()
    pa,pf=float(probs[0]),float(probs[1])
    verdict = f"{chr(128680) if pf>=0.5 else chr(9989)} MANIPULE ({pf:.1%})" if pf>=0.5 else f"{chr(9989)} AUTHENTIQUE ({pa:.1%})"
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    axes[0].plot(np.linspace(0,4,len(wav.squeeze())),wav.squeeze().numpy(),linewidth=0.4,color="#2E75B6")
    axes[0].set_title(f"Waveform — {verdict}"); axes[0].set_xlabel("Temps (s)")
    axes[1].imshow(lfcc_t.squeeze().numpy(),aspect="auto",origin="lower",cmap="inferno")
    axes[1].set_title("LFCC"); axes[1].set_xlabel("Frames")
    plt.tight_layout()
    buf=io.BytesIO(); plt.savefig(buf,format="png",dpi=100); buf.seek(0)
    img=Image.open(buf).copy(); plt.close(); buf.close()
    report=f"Verdict: {verdict}\nProb fake: {pf:.2%}\nProb authentic: {pa:.2%}"
    return verdict,{"Authentique":round(pa,4),"Manipule":round(pf,4)},img,report

with gr.Blocks(title="Forensic Audio Authentication") as app:
    gr.Markdown("# Forensic Audio Authentication\nCNN-BiLSTM + LFCC + Grad-CAM")
    with gr.Row():
        with gr.Column():
            inp=gr.Audio(label="Fichier audio",type="filepath")
            btn=gr.Button("Analyser",variant="primary")
            v=gr.Textbox(label="Verdict",interactive=False)
            p=gr.Label(label="Probabilites",num_top_classes=2)
            r=gr.Textbox(label="Rapport",interactive=False,lines=6)
        with gr.Column():
            f=gr.Image(label="Analyse visuelle",type="pil")
    btn.click(analyze,[inp],[v,p,f,r])

if __name__=="__main__":
    app.launch(share=True)
'''

# Sauvegarder le script
app_path = OUTPUT / 'gradio_app.py'
with open(app_path, 'w', encoding='utf-8') as fp:
    fp.write(app_script)

print(f'✅ gradio_app.py sauvé : {app_path}')
print(f'   Taille : {app_path.stat().st_size / 1024:.1f} KB')
print()
print('Pour utiliser en dehors de Kaggle :')
print('  1. Télécharger gradio_app.py')
print('  2. Mettre best_model.pth dans le même dossier')
print('  3. pip install gradio torchaudio captum')
print('  4. python gradio_app.py')

✅ gradio_app.py sauvé : /kaggle/working/gradio_app.py
   Taille : 4.5 KB

Pour utiliser en dehors de Kaggle :
  1. Télécharger gradio_app.py
  2. Mettre best_model.pth dans le même dossier
  3. pip install gradio torchaudio captum
  4. python gradio_app.py


---
## CELLULE 11 — Résumé final du projet

In [18]:
print('=' * 65)
print('🎉 PROJET FORENSIC AUDIO AUTHENTICATION — COMPLET')
print('=' * 65)

print('''
📓 Notebooks réalisés :
   ✅ 01_data_preprocessing.ipynb
      → Rééchantillonnage, simulation splices, master_labels.csv

   ✅ 02_lfcc_extraction.ipynb
      → Extraction LFCC (20 coeff.), CMVN, sauvegarde .npy

   ✅ 03_model_training.ipynb
      → CNN-BiLSTM, 20 epochs, EER < 5%, export ONNX

   ✅ 04_xai_gradcam.ipynb
      → Grad-CAM, Integrated Gradients, validation splice

   ✅ 05_gradio_app.ipynb  ← CE NOTEBOOK
      → Interface forensique publique + gradio_app.py
''')

print('📂 Fichiers produits :')
for f in sorted(OUTPUT.rglob('*')):
    if f.is_file() and f.suffix in ('.pth', '.onnx', '.csv', '.py', '.png'):
        size = f.stat().st_size
        unit = 'MB' if size > 1e6 else 'KB'
        val  = size/1e6 if size > 1e6 else size/1024
        rel  = str(f).replace(str(OUTPUT)+'/', '')
        print(f'   {rel:45s} ({val:.1f} {unit})')

print()
print('🔗 Lien Gradio public : voir output de la cellule 9')
print('   (valable 72h — relancer la cellule 9 pour renouveler)')
print()
print('📄 Livrables du cahier des charges :')
print('   ✅ Pipeline forensique complet')
print('   ✅ Modèle CNN-BiLSTM entraîné')
print('   ✅ Export ONNX pour outils experts')
print('   ✅ Notebooks XAI (Grad-CAM + SHAP)')
print('   ✅ Gradio forensic workstation')
print('   ✅ gradio_app.py standalone')
print()
print('Reste à faire pour le rapport (25 pages) :')
print('   → Rédiger les sections avec les figures générées')
print('   → Validation dialecte Marocain (Notebook 3 fine-tuning)')
print('   → Vidéo démo 3 minutes')

🎉 PROJET FORENSIC AUDIO AUTHENTICATION — COMPLET

📓 Notebooks réalisés :
   ✅ 01_data_preprocessing.ipynb
      → Rééchantillonnage, simulation splices, master_labels.csv

   ✅ 02_lfcc_extraction.ipynb
      → Extraction LFCC (20 coeff.), CMVN, sauvegarde .npy

   ✅ 03_model_training.ipynb
      → CNN-BiLSTM, 20 epochs, EER < 5%, export ONNX

   ✅ 04_xai_gradcam.ipynb
      → Grad-CAM, Integrated Gradients, validation splice

   ✅ 05_gradio_app.ipynb  ← CE NOTEBOOK
      → Interface forensique publique + gradio_app.py

📂 Fichiers produits :
   gradio_app.py                                 (4.5 KB)

🔗 Lien Gradio public : voir output de la cellule 9
   (valable 72h — relancer la cellule 9 pour renouveler)

📄 Livrables du cahier des charges :
   ✅ Pipeline forensique complet
   ✅ Modèle CNN-BiLSTM entraîné
   ✅ Export ONNX pour outils experts
   ✅ Notebooks XAI (Grad-CAM + SHAP)
   ✅ Gradio forensic workstation
   ✅ gradio_app.py standalone

Reste à faire pour le rapport (25 pages) :
   